# Multi-Label Classification

This notebook trains and evaluates multi-label classification models for plant health assessment.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src to path
sys.path.append('../src')

from models import get_model
from data_loader import get_data_loaders
from train import Trainer
from evaluate import MultiLabelEvaluator

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

plt.rcParams['figure.figsize'] = (12, 8)

## Load Data

In [ ]:
# Load data loaders
data_loaders = get_data_loaders(
    data_dir='../data/raw',
    metadata_file='../data/raw/metadata.csv',
    batch_size=32,
    num_workers=4,
    image_size=224
)

print(f"Train batches: {len(data_loaders['train'])}")
print(f"Validation batches: {len(data_loaders['val'])}")
print(f"Test batches: {len(data_loaders['test'])}")

## Initialize Model

In [ ]:
# Create multi-label model
model = get_model(
    model_type='multi_label',
    backbone='efficientnet_b0',
    num_diseases=7,
    num_pests=5,
    num_abiotic=5,
    pretrained=True
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Test Forward Pass

In [ ]:
# Test forward pass
model.eval()
with torch.no_grad():
    dummy_input = torch.randn(2, 3, 224, 224).to(device)
    outputs = model(dummy_input)
    
    print("Output keys:", outputs.keys())
    for key, value in outputs.items():
        print(f"  {key}: {value.shape}")

## Training Setup

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    train_loader=data_loaders['train'],
    val_loader=data_loaders['val'],
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-5,
    log_dir='../experiments/logs/multi_label'
)

print("Trainer initialized successfully")

## Training (Uncomment to run)

In [ ]:
# Train the model
# trainer.train(num_epochs=50, save_dir='../experiments/checkpoints')

## Load Trained Model (if available)

In [ ]:
# Load checkpoint if available
checkpoint_path = '../experiments/checkpoints/best_model.pth'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Model loaded from {checkpoint_path}")
    print(f"Validation loss: {checkpoint['val_loss']:.4f}")
else:
    print("No checkpoint found. Please train the model first.")

## Evaluation

In [ ]:
# Initialize evaluator
evaluator = MultiLabelEvaluator(model, device=device)

# Evaluate on test set
metrics = evaluator.evaluate(data_loaders['test'], threshold=0.5)

# Print metrics
print("\n" + "="*50)
print("Evaluation Metrics")
print("="*50)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key:30s}: {value:.4f}")

## Per-Class Analysis

In [ ]:
# Label names
label_names = [
    'early_blight', 'late_blight', 'powdery_mildew', 'leaf_spot',
    'bacterial_spot', 'viral_infection', 'fungal_infection',
    'aphids', 'whiteflies', 'thrips', 'spider_mites', 'caterpillars',
    'nutrient_deficiency', 'water_stress', 'heat_stress',
    'salt_stress', 'light_stress'
]

# Get per-class metrics
per_class_metrics = evaluator.evaluate_per_class(data_loaders['test'], label_names)

# Display per-class metrics
print("\nPer-Class Metrics:")
print("="*80)
for label, metrics_dict in per_class_metrics.items():
    print(f"\n{label}:")
    print(f"  Precision: {metrics_dict['precision']:.4f}")
    print(f"  Recall: {metrics_dict['recall']:.4f}")
    print(f"  F1: {metrics_dict['f1']:.4f}")
    print(f"  Support: {metrics_dict['support']}")

## Visualize Metrics

In [ ]:
# Extract F1 scores for visualization
f1_scores = [metrics_dict['f1'] for metrics_dict in per_class_metrics.values()]
supports = [metrics_dict['support'] for metrics_dict in per_class_metrics.values()]

# Plot F1 scores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(label_names, f1_scores, color='#3498db')
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Per-Class F1 Scores')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5)

axes[1].barh(label_names, supports, color='#2ecc71')
axes[1].set_xlabel('Support (Number of Samples)')
axes[1].set_title('Per-Class Support')

plt.tight_layout()
plt.show()

## Prediction Analysis

In [ ]:
# Analyze prediction distribution
def analyze_predictions(data_loader, model, device, threshold=0.5):
    """Analyze prediction distributions"""
    model.eval()
    all_probs = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            
            if 'multi_label' in outputs:
                probs = torch.sigmoid(outputs['multi_label']).cpu().numpy()
                preds = (probs >= threshold).astype(int)
                true = labels['multi_label'].numpy()
            elif 'diseases' in outputs:
                disease_probs = torch.sigmoid(outputs['diseases']).cpu().numpy()
                pest_probs = torch.sigmoid(outputs['pests']).cpu().numpy()
                abiotic_probs = torch.sigmoid(outputs['abiotic']).cpu().numpy()
                probs = np.concatenate([disease_probs, pest_probs, abiotic_probs], axis=1)
                preds = (probs >= threshold).astype(int)
                true = labels['multi_label'].numpy()
            
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(true)
    
    all_probs = np.vstack(all_probs)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    return all_probs, all_preds, all_labels

# Analyze predictions
probs, preds, labels = analyze_predictions(data_loaders['test'], model, device)

# Calculate average number of labels per sample
avg_pred_labels = preds.sum(axis=1).mean()
avg_true_labels = labels.sum(axis=1).mean()

print(f"Average predicted labels per sample: {avg_pred_labels:.2f}")
print(f"Average true labels per sample: {avg_true_labels:.2f}")

## Summary

In [ ]:
print("Multi-Label Classification Summary:")
print("- Model architecture: Multi-head EfficientNet")
print("- Tasks: Disease, pest, and abiotic stress classification")
print("- Additional outputs: Health status and severity estimation")
print("- Evaluation metrics: Precision, Recall, F1, Hamming loss")
print("- Per-class analysis helps identify weak classes")